In [ ]:
import pennylane as qml
import jax

n_qubits = 4  # 量子比特数
n_layers = 1  # 量子电路层数

# 创建支持JAX的量子设备
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface='jax')
def quantum_circuit(params,s):
    """参数化量子电路生成复波函数"""
    # 编码经典输入s到量子态
    for i in range(n_qubits):
        if s[i] == 1:  # |1>态
            qml.PauliX(wires=i)
    
    # 硬件高效拟设
    for layer in range(n_layers):
        # 单量子比特旋转层
        for i in range(n_qubits):
            qml.RY(params[layer, i, 0], wires=i)
            qml.RZ(params[layer, i, 1], wires=i)
        
        # 两量子比特纠缠层
        for i in range(0, n_qubits - 1, 2):  # 相邻量子比特纠缠
            qml.CNOT(wires=[i, i + 1])
        for i in range(1, n_qubits - 1, 2):  # 交错纠缠模式
            qml.CNOT(wires=[i, i + 1])
        
        # 第二层单量子比特旋转
        for i in range(n_qubits):
            qml.RY(params[layer, i, 2], wires=i)
            qml.RZ(params[layer, i, 3], wires=i)
    
    # 返回状态向量
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]




In [29]:
params = np.random.rand(1, 4, 4)

params.shape

(1, 4, 4)

In [32]:
import numpy as np
print(qml.draw(quantum_circuit)(params, s=[1, 0, 1, 0]))

# 生成所有可能的构型

0: ──X─────────RY(0.31)──RZ(0.92)─╭●──RY(0.46)──RZ(0.44)───────────┤  <Z>
1: ──RY(0.17)──RZ(0.04)───────────╰X─╭●─────────RY(0.65)──RZ(0.33)─┤  <Z>
2: ──X─────────RY(0.17)──RZ(0.66)─╭●─╰X─────────RY(0.83)──RZ(0.04)─┤  <Z>
3: ──RY(0.39)──RZ(0.95)───────────╰X──RY(0.53)──RZ(0.88)───────────┤  <Z>
